# Section 6 — CPU float64 gate

Only the registered CPU checks. Exact references and relative comparisons run again. InfSSM remains DEFERRED until phase two.

Same setup flow as H4: Internet ON, Kaggle Secret `github_token`, T4 x2. Source is cloned from GitHub and checked out at a fixed commit. Attach raw ImageNet-R (for example my1nonly/imagenet-r) and official pretrained weights. The notebook creates train/val symlinks using the pinned Mamba-CL lists, checks all24000/6000 files and200 classes, and also accepts an existing official split. Previous Output is for resume and for carrying01 selection into02. No source ZIP is required.

Active CPU checks pass and are recomputed. CUDA compatibility, VRAM and throughput remain unmeasured.

In [ ]:
BRANCH = 'GL'
REPO_COMMIT = 'fdc14b7a0bb30425622bb194d2e863dc8608fd16'
RUN_TAG = 'glcl_imagenetr_s0_fdc14b7a0bb3'
STAGE = 'tests'
DATA_ROOT = None  # optional raw OR official train/val root; auto-discovered under /kaggle/input
PRETRAINED = None  # optional explicit path to defocus_mamba_large_cls_21k.pth
JOBS = [[], []]
import time
SESSION_START = time.time()

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

os.environ.update(TQDM_DISABLE='1', PYTHONUNBUFFERED='1',
                  PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python')
MAIN_URL = 'https://github.com/duyh80456-code/Gramian-Laplace-GL-cho-continual-learning-tr-n-Mamba.git'
REPO = Path('/kaggle/working/gramian-laplace')
RUN_ROOT = Path('/kaggle/working/glcl')
OUTPUT = RUN_ROOT / 'glcl_outputs'
ROOT_OUT = OUTPUT

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def private_clone(url, destination, commit):
    token = UserSecretsClient().get_secret('github_token').strip()
    if not token:
        raise RuntimeError('Enable Kaggle Secret github_token for this notebook')
    askpass = Path('/kaggle/working/.glcl_git_askpass.py')
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = dict(os.environ, GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
    try:
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)], env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'fetch', 'origin', commit], env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

# Only refresh the source checkout. Outputs and environment live elsewhere.
if REPO.exists():
    shutil.rmtree(REPO)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if CODE_COMMIT != REPO_COMMIT:
    raise RuntimeError('Pinned repository revision mismatch')
os.chdir(REPO)
print('Resolved code commit:', CODE_COMMIT, 'pin:', REPO_COMMIT)
print('Official MambaCL:', json.loads((REPO/'upstream/LOCK.json').read_text()))

In [ ]:
# Isolated Python 3.11 + official torch 2.2.2; no Kaggle host torch replacement.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.5.31'], check=True)
VENV = Path('/kaggle/working/glcl_env')
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                '-r', str(REPO/'requirements-kaggle.txt'), 'pip==24.2'], check=True)

ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1', TQDM_DISABLE='1')

In [ ]:
# Recompute the supervisor-corrected section-six checks, same fixture/noises:
# C .852433 > proxy .725509 and >=.8; Delta .968065 > proxy .950543 and >=.85.
# Test5 remains DEFERRED until phase two.
# A failure intentionally stops before model initialization/training.
subprocess.run([PYTHON, '-m', 'tests.section6'], env=ENV, check=True)
print((REPO/'reports/section6.json').read_text())